In [22]:
import requests
import pandas as pd
import sqlite3

In [4]:
def get_buienradar_measurements(
    return_type: str = "json",
):
    try:
        url = f"https://data.buienradar.nl/2.0/feed/{return_type}"
    except ValueError:
        raise ValueError("Invalid return type. Supported types are 'json' and 'xml'.")

    response = requests.get(url)
    response.raise_for_status()

    return response.json()["actual"]

In [7]:
actual_data = get_buienradar_measurements(return_type="json")

In [13]:
columns = [
    "timestamp",
    "temperature",
    "groundtemperature",
    "feeltemperature",
    "windgusts",
    "windspeedBft",
    "humidity",
    "precipitation",
    "sunpower",
    "stationid",
]

df_measurements = pd.json_normalize(actual_data["stationmeasurements"])[columns]

# Create a unique measurement_id by combining stationid and timestamp, could also go uuid but this is more informative
df_measurements["measurement_id"] = df_measurements["stationid"].astype(str) + "_" + df_measurements["timestamp"]
df_measurements = df_measurements[["measurement_id"] + columns]

In [14]:
df_measurements

,measurement_id,timestamp,temperature,groundtemperature,feeltemperature,windgusts,windspeedBft,humidity,precipitation,sunpower,stationid
0,6275_2026-04-26T13:40:00,2026-04-26T13:40:00,14.2,18.1,14.2,5.8,2.0,36.0,0.0,814.0,6275
1,6249_2026-04-26T13:40:00,2026-04-26T13:40:00,13.0,17.8,13.0,3.9,1.0,61.0,0.0,825.0,6249
2,6260_2026-04-26T13:40:00,2026-04-26T13:40:00,14.3,18.8,14.3,4.6,2.0,43.0,0.0,812.0,6260
3,6235_2026-04-26T13:40:00,2026-04-26T13:40:00,10.9,14.2,9.2,5.4,3.0,60.0,0.0,835.0,6235
4,6370_2026-04-26T13:40:00,2026-04-26T13:40:00,14.4,17.8,14.4,4.9,2.0,43.0,0.0,829.0,6370
5,6377_2026-04-26T13:40:00,2026-04-26T13:40:00,15.3,19.0,15.3,5.1,2.0,40.0,0.0,818.0,6377
6,6350_2026-04-26T13:40:00,2026-04-26T13:40:00,14.8,19.9,14.8,4.1,2.0,45.0,0.0,718.0,6350
7,6323_2026-04-26T13:40:00,2026-04-26T13:40:00,13.3,15.9,12.0,6.1,3.0,58.0,0.0,842.0,6323
8,6283_2026-04-26T13:40:00,2026-04-26T13:40:00,13.6,16.8,13.6,4.0,2.0,42.0,0.0,853.0,6283
9,6280_2026-04-26T13:40:00,2026-04-26T13:40:00,12.5,16.7,12.5,5.0,2.0,48.0,0.0,816.0,6280


In [15]:
df_measurements.to_csv("buienradar_measurements.csv", index=False)

In [20]:
df_stations = pd.DataFrame(actual_data["stationmeasurements"])[["stationid", "stationname", "lat", "lon", "regio"]]

In [21]:
df_stations.to_csv("buienradar_stations.csv", index=False)

In [28]:
def init_db():
    with sqlite3.connect("buienradar.db") as conn:
        cursor = conn.cursor()

        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS stations (
                stationid INTEGER PRIMARY KEY,
                stationname TEXT,
                lat REAL,
                lon REAL,
                regio TEXT
            )
            """
        )

        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS measurements (
                measurementid INTEGER PRIMARY KEY,
                timestamp TEXT,
                groundtemperature REAL,
                feeltemperature REAL,
                windgusts REAL,
                windspeedBft REAL,
                humidity REAL,
                precipitation REAL,
                sunpower REAL,
                stationid INTEGER REFERENCES stations(stationid)
            )
            """
        )


In [29]:
init_db()